# Advanced Iterative Feature Engineering & Incremental Validation Pipelines

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from src import feature_engineering as fe

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Environment Configurations & Partition Loading

In [2]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_val = pd.read_csv("../data/processed/raw_features/X_val.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")

y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_val = pd.read_csv("../data/processed/target/y_val.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [3]:
# Operational Path Anchors
ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")
EXPERIMENT_NAME = "customer-churn-simple-feature-engineering"

In [4]:
# Schema Baseline Columns Definitions
nomod_columns = [
    'HasCrCard',
    'IsActiveMember'
    ]

dummyfy_columns = [
    'Card Type', 
    'NumOfProducts', 
    'Geography', 
    'Gender'
    ]

norm_std_columns = [
    'Balance', 
    'Point Earned', 
    'CreditScore', 
    'Age', 
    'Tenure', 
    'Satisfaction Score', 
    'EstimatedSalary'
    ]

In [5]:

# Candidates List generated by Custom Transformer Pipeline Block
new_feature_engineered_columns = [
    "no_balance", 
    "NumOfProducts_1",
    "NumOfProducts_2", 
    "Balance_x_Tenure", 
    "Age_x_IsActive",
    "Balance_to_Salary", 
    "Balance_per_Product", 
    "Salary_per_Product", 
    "CreditScore_per_Age", 
    "Tenure_per_Age",
    "Inactive_x_Balance", 
    "Inactive_x_Age", 
    "Products_x_Active", 
    "Balance_plus_Salary", 
    "WealthScore",
    "CreditScore_x_Age", 
    "LogBalance", 
    "LogAge", 
    "Age2", 
    "Balance2", 
    "Tenure2", 
    "Products_per_Tenure", 
    "Balance_per_Tenure"
    ]

## 2. Global Multi-Model Experiment Log Execution

In [6]:
fe.init_mlflow_experiment(EXPERIMENT_NAME, DB_PATH, ARTIFACTS_DIR)

In [7]:
# Combined Preprocessing Strategy Wrapper Block
global_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), dummyfy_columns),
    ('num', StandardScaler(), norm_std_columns + new_feature_engineered_columns),
    ('pass', 'passthrough', nomod_columns)
], remainder='drop')

models_zoo = {
    "random_forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "xgboost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric="aucpr")
}

In [8]:
# Run Experiment Training Execution Loop
fe.train_and_log_pipeline(
    models=models_zoo, preprocessor=global_preprocessor,
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    input_feature_list=dummyfy_columns + norm_std_columns + nomod_columns
)

2026/06/02 17:03:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished Logging Pipeline Execution: random_forest | PR-AUC: 0.7181


2026/06/02 17:03:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished Logging Pipeline Execution: xgboost | PR-AUC: 0.7061


In [9]:
# Review Tracking Telemetry Results Matrix
fe.get_experiment_summary(EXPERIMENT_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc,start_time
1,8928c58fe1b34762949017fa4b06f9b2,random_forest,0.8750,0.813492,0.502451,0.621212,0.867759,0.718064,2026-06-02 23:03:41.075000+00:00
0,1e71bbb22897461ebf3c47a2bd9a63b4,xgboost,0.8645,0.740351,0.517157,0.608947,0.858993,0.706053,2026-06-02 23:03:44.080000+00:00


## 3. Incremental Feature Assessment (Greedy Contribution Evaluation Search)

In [10]:
# Evaluate standalone contribution performance changes against baseline model references
search_results_df = fe.evaluate_engineered_features(
    engineered_features=new_feature_engineered_columns,
    base_nomod_columns=nomod_columns,
    dummyfy_columns=dummyfy_columns,
    norm_std_columns=norm_std_columns,
    model=models_zoo["random_forest"],
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test
)

display(search_results_df)

,feature_added,pr_auc
0,BASELINE,0.733387
1,Age_x_IsActive,0.732918
2,NumOfProducts_1,0.731978
3,Inactive_x_Age,0.731502
4,Balance_per_Product,0.730197
5,Tenure2,0.729435
6,Salary_per_Product,0.729184
7,Inactive_x_Balance,0.727405
8,Balance_per_Tenure,0.727174
9,Products_x_Active,0.727110
